In [1]:
import torch
inputs = torch.tensor(
    [[
        [0.43, 0.15, 0.89], # Your     
        [0.55, 0.87, 0.66], # journey  
        [0.57, 0.85, 0.64], # starts   
        [0.22, 0.58, 0.33], # with     
        [0.77, 0.25, 0.10], # one      
        [0.05, 0.80, 0.55]  # step     
    ],
    [
        [0.43, 0.15, 0.89], # His     
        [0.55, 0.87, 0.66], # journey  
        [0.57, 0.85, 0.64], # end   
        [0.22, 0.58, 0.33], # at     
        [0.77, 0.25, 0.10], # last      
        [0.05, 0.80, 0.55]  # way     
    ]
])

In [2]:

import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,num_heads, dropout, qkv_bias = False):
        super().__init__()
        
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
        
        # shape
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dims = d_out // num_heads

        # Wq , Wk, Wv 
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.dropout = nn.Dropout(dropout)
        
        # Linear layer for combine head outputs
        self.output_proj = nn.Linear(d_out, d_out)
        
        
        self.register_buffer(
            "mask",
            torch.triu( torch.ones(context_length,context_length) , diagonal=1 )
        )
        
    def forward(self,x):
        no_of_batch, no_of_token, embedding_dim = x.shape
        
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        queries = queries.view(
            no_of_batch, no_of_token, self.num_heads, self.head_dims
        )
        keys = keys.view(no_of_batch, no_of_token, self.num_heads, self.head_dims)
        values = values.view(no_of_batch, no_of_token, self.num_heads, self.head_dims)
        
        queries = queries.transpose(1,2)
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)
        
        # attention = Q @ (K)T
        attention_score = queries @ keys.transpose(2,3)
        
        # masking
        attention_score = attention_score.masked_fill(
            self.mask.bool()[:no_of_token, :no_of_token], -torch.inf
        )
        
        # normalize
        d_k = keys.shape[-1]
        attention_weight = torch.softmax(
            attention_score/d_k**0.5, dim=-1
        )
        
        # dropout
        attention_weight = self.dropout(attention_weight)
        
        context_vector = (attention_weight @ values).transpose(1,2)
        context_vector = context_vector.contiguous().view(
            no_of_batch, no_of_token, self.d_out
        )
        context_vector = self.output_proj(context_vector)
        return context_vector

In [3]:
d_in = inputs.shape[-1]
d_out = 2
context_length = inputs.shape[1]
print(d_in, d_out, context_length)

3 2 6


In [4]:
mha = MultiHeadAttention(d_in,d_out,context_length, dropout=0.0, num_heads=2)

In [5]:
context_vector = mha(inputs)
context_vector

tensor([[[ 0.5571, -0.2144],
         [ 0.5454, -0.1746],
         [ 0.5418, -0.1613],
         [ 0.5365, -0.1628],
         [ 0.5294, -0.1597],
         [ 0.5304, -0.1618]],

        [[ 0.5571, -0.2144],
         [ 0.5454, -0.1746],
         [ 0.5418, -0.1613],
         [ 0.5365, -0.1628],
         [ 0.5294, -0.1597],
         [ 0.5304, -0.1618]]], grad_fn=<ViewBackward0>)

In [6]:
context_vector.shape

torch.Size([2, 6, 2])